# Notebook 01: Synthetic Study
**Run order:** This notebook should be run first.

What it does:
1. Installs packages
2. Clones/uploads the project (see Step 2 instructions)
3. Generates the four canonical synthetic datasets (n=1000 and n=50000, biased and unbiased)
4. Runs all six causal-discovery algorithms on each dataset
5. Saves results to `results/synthetic_results.pkl`
6. Prints a summary table

**Estimated runtime:** ~5 minutes on Colab CPU (n=50000 DirectLiNGAM is the slowest step)

## Step 1 — Install packages

In [ ]:
# Install all required packages
# This takes ~2-3 minutes on first run; Colab caches after that
!pip install -q causal-learn dowhy econml networkx matplotlib numpy pandas scipy scikit-learn seaborn joblib tqdm

## Step 2 — Upload or clone the project

**Option A (recommended): Clone from GitHub after you push the project there**
```python
!git clone https://github.com/YOUR_USERNAME/causal_bias_project.git
%cd causal_bias_project
```

**Option B: Upload a zip file**
```python
from google.colab import files
uploaded = files.upload()   # upload causal_bias_project.zip
!unzip -q causal_bias_project.zip
%cd causal_bias_project
```

**Option C: Mount Google Drive (if you saved it there)**
```python
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/causal_bias_project
```

Run whichever option applies, then continue with Step 3.

In [ ]:
# ── CHOOSE ONE of the options above, then run this to verify ──
import os
# If you're already in the right directory, this should show your project files
print(os.listdir('.'))

# Add project root to path so imports work
import sys
sys.path.insert(0, os.getcwd())
print('Project root on sys.path:', os.getcwd())

## Step 3 — Generate canonical datasets

In [ ]:
import numpy as np
import pandas as pd
from synthetic.scm import (
    generate_loan_data, standardize, ground_truth_adj,
    CANONICAL_N, VALIDATION_N, CANONICAL_SEED,
    BETA_BIASED, BETA_UNBIASED, VAR_NAMES
)

# Generate all four canonical datasets
datasets = {
    'n1000_biased':    generate_loan_data(CANONICAL_N,  BETA_BIASED,   CANONICAL_SEED),
    'n1000_unbiased':  generate_loan_data(CANONICAL_N,  BETA_UNBIASED, CANONICAL_SEED),
    'n50000_biased':   generate_loan_data(VALIDATION_N, BETA_BIASED,   CANONICAL_SEED),
    'n50000_unbiased': generate_loan_data(VALIDATION_N, BETA_UNBIASED, CANONICAL_SEED),
}

os.makedirs('results', exist_ok=True)
for label, X in datasets.items():
    np.save(f'results/synthetic_{label}.npy', X)

# Print disparity check
for label, X in datasets.items():
    df = pd.DataFrame(X, columns=VAR_NAMES)
    r0 = df[df['Race']==0]['LoanApprv'].mean()
    r1 = df[df['Race']==1]['LoanApprv'].mean()
    print(f'{label:25s}  n={len(df):6d}  '
          f'Approval Race=0: {r0:.3f}  Race=1: {r1:.3f}  '
          f'DIR={r1/r0:.3f}')

print('\n✓ Canonical datasets generated and saved.')

## Step 4 — Run all algorithms

In [ ]:
# Run all 6 algorithms on all 4 datasets
# This cell calls the experiments module directly
import importlib
import experiments.run_synthetic as run_syn
importlib.reload(run_syn)

df_results = run_syn.main()
print(f'\nTotal results rows: {len(df_results)}')

## Step 5 — Summary table

In [ ]:
import pandas as pd

df_results = pd.read_pickle('results/synthetic_results.pkl')

# Pivot to paper-style table
summary = df_results[['dataset','algorithm','shd','detected','beta_hat']].copy()
summary['beta_hat'] = summary['beta_hat'].apply(
    lambda x: f'{x:+.4f}' if (x is not None and not pd.isna(x)) else '—'
)
summary['detected'] = summary['detected'].map({True: '✓ YES', False: 'no'})

for dataset in ['n1000_biased','n1000_unbiased','n50000_biased','n50000_unbiased']:
    print(f'\n── {dataset} ─────────────────────────────────────')
    sub = summary[summary['dataset']==dataset][['algorithm','shd','detected','beta_hat']]
    sub.columns = ['Algorithm','SHD','Race→Loan','β̂']
    print(sub.to_string(index=False))

## Step 6 — Generate Figure 1 (ground truth DAG) and Figure 3 (DirectLiNGAM)
Run figure generation here or in notebook 04.

In [ ]:
os.makedirs('figures/output', exist_ok=True)

# Figure 1
import figures.fig_gt_biased as f1
importlib.reload(f1)
f1.make_figure()

# Figure 3
import figures.fig_directlingam_loan as f3
importlib.reload(f3)
f3.make_figure()

print('\nFigures saved to figures/output/')
print(os.listdir('figures/output/'))

In [ ]:
# Preview the PDFs in notebook as PNG
from IPython.display import Image
Image('figures/output/fig_gt_biased.png')

In [ ]:
Image('figures/output/fig_directlingam_loan.png')